# Uni-Mol 5-class odor classification — Colab-safe LoRA fine-tuning

In [5]:
%pip install -q scikit-learn unimol-tools rdkit

In [6]:
from pathlib import Path
import subprocess

repo_dir = Path('molecular-foundation-model')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/FufanLu/molecular-foundation-model.git',
        str(repo_dir),
    ], check=True)
else:
    print(f'{repo_dir} already exists; reusing it.')

import sys
sys.path.insert(0, str(repo_dir.resolve()))

from src.dataset.load_leffingwell import load_leffingwell
from src.preprocessing.clean_smiles import clean_smiles
from src.classifier.label_encoder import encode_labels, label_distribution, ALL_5_CLASSES

df = encode_labels(clean_smiles(load_leffingwell()))
print(f'{len(df)} valid molecules')
label_distribution(df)

molecular-foundation-model already exists; reusing it.
3522 -> 3522 valid molecules (0 removed)
3522 valid molecules
Class         Count
--------------------
fruity         2353
sweet          1497
sulfurous       921
woody           968
fatty           749
--------------------
Multi-label:  2281
Total:        3522


## Prepare Uni-Mol inputs once

In [7]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from unimol_tools.data import DataHub

torch.set_num_threads(2)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

smiles_list = df['smiles'].tolist()
print(f'Generating 3D inputs for {len(smiles_list)} molecules...')

# IMPORTANT: DataHub/ConformerGen defaults to multiprocessing on Linux.
# Forking after CUDA has been initialized is a common Colab deadlock.
hub = DataHub(
    data=smiles_list,
    task='repr',
    is_train=False,
    data_type='molecule',
    model_name='unimolv1',
    batch_size=4,
    remove_hs=False,
    use_cuda=torch.cuda.is_available(),
    use_ddp=False,
    use_gpu='0',
    multi_process=False,
)
features = hub.data['unimol_input']
Y = np.stack(df['y'].values).astype(np.float32)
indices = np.arange(len(features))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
print(f'Prepared {len(features)} inputs; train={len(train_idx)}, test={len(test_idx)}')

Device: cuda:0
NVIDIA A100-SXM4-40GB
Generating 3D inputs for 3522 molecules...


2026-07-15 02:00:20 | unimol_tools/weights/weighthub.py | 54 | INFO | Uni-Mol Tools | Weights will be downloaded to default directory: /usr/local/lib/python3.12/dist-packages/unimol_tools/weights
INFO:Uni-Mol Tools:Weights will be downloaded to default directory: /usr/local/lib/python3.12/dist-packages/unimol_tools/weights
2026-07-15 02:00:20 | unimol_tools/weights/weighthub.py | 71 | INFO | Uni-Mol Tools | Downloading mol.dict.txt
INFO:Uni-Mol Tools:Downloading mol.dict.txt
DEBUG:httpcore.connection:connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=3 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7d7d6d8abbc0>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x7d7d6d74a2d0> server_hostname='huggingface.co' timeout=3
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7d7d6d8acda0>
DEBUG:httpc

DEBUG:filelock:Attempting to acquire lock 137977662272768 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

DEBUG:filelock:Lock 137977662272768 acquired on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock
DEBUG:filelock:Attempting to release lock 137977662272768 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock
DEBUG:filelock:Lock 137977662272768 released on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock
DEBUG:filelock:Attempting to acquire lock 137977662269168 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/download/mol.dict.txt.lock
DEBUG:filelock:Lock 137977662269168 acquired on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/download/mol.dict.txt.lock
DEBUG:filelock:Attempting to release lock 137977662269168 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/download/mol.dict.txt.lock
DEBUG:filelock:Lock 137977662269168 released on /usr/local/lib/p

Prepared 3522 inputs; train=2817, test=705


## Build a differentiable Uni-Mol + LoRA model

In [13]:
import torch.nn as nn
from unimol_tools.models import UniMolModel
from src.classifier.model import apply_lora

backbone = UniMolModel(
    output_dim=len(ALL_5_CLASSES),
    data_type='molecule',
    remove_hs=False,
).to(device)

# Freeze pretrained weights first. apply_lora adds trainable adapters.
for parameter in backbone.parameters():
    parameter.requires_grad = False

attention_linear_names = [
    name for name, module in backbone.named_modules()
    if isinstance(module, nn.Linear) and name.endswith('in_proj')
]
if not attention_linear_names:
    raise RuntimeError('No Uni-Mol attention in_proj layers found.')
# Adapting only the last four blocks is much faster on Colab while
# preserving genuine gradient-based Uni-Mol fine-tuning.
target_names = attention_linear_names[-4:]
print('LoRA targets:', target_names)

n_replaced = apply_lora(backbone, r=4, alpha=8, target_names=target_names)
# apply_lora may create new parameters; normalize every parameter device.
backbone = backbone.to(device)
for parameter in backbone.classification_head.parameters():
    parameter.requires_grad = True
if n_replaced == 0:
    raise RuntimeError('LoRA matched zero layers.')

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total = sum(p.numel() for p in backbone.parameters())
print(f'LoRA layers replaced: {n_replaced}')
print(f'Trainable parameters: {trainable:,} / {total:,}')

INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[PAD]', index is 2.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[CLS]', index is 0.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[SEP]', index is 3.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[UNK]', index is 1.
2026-07-15 02:11:55 | unimol_tools/models/unimol.py | 169 | INFO | Uni-Mol Tools | Loading pretrained weights from /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/mol_pre_all_h_220816.pt
INFO:Uni-Mol Tools:Loading pretrained weights from /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/mol_pre_all_h_220816.pt


LoRA targets: ['encoder.layers.11.self_attn.in_proj', 'encoder.layers.12.self_attn.in_proj', 'encoder.layers.13.self_attn.in_proj', 'encoder.layers.14.self_attn.in_proj']
LoRA layers replaced: 4
Trainable parameters: 35,333 / 47,365,959


## Train (no `get_repr` inside the loop)

In [14]:
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

class MoleculeDataset(Dataset):
    def __init__(self, inputs, labels, indices):
        self.inputs = inputs
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = int(self.indices[position])
        return self.inputs[index], self.labels[index]

train_loader = DataLoader(
    MoleculeDataset(features, Y, train_idx),
    batch_size=8, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available(),
    collate_fn=backbone.batch_collate_fn,
)
test_loader = DataLoader(
    MoleculeDataset(features, Y, test_idx),
    batch_size=8, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available(),
    collate_fn=backbone.batch_collate_fn,
)

optimizer = torch.optim.AdamW(
    [p for p in backbone.parameters() if p.requires_grad],
    lr=2e-4, weight_decay=1e-4,
)
criterion = nn.BCEWithLogitsLoss()
epochs = 30
amp_enabled = device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
checkpoint_path = 'odor_lora_checkpoint.pt'
start_epoch = 0
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if checkpoint.get('target_names') == target_names:
        backbone.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = int(checkpoint['epoch']) + 1
        print(f'Resuming from epoch {start_epoch + 1}')
    else:
        print('Ignoring checkpoint with a different LoRA target set.')

def evaluate(loader):
    backbone.eval()
    logits, labels = [], []
    with torch.no_grad():
        for batch_inputs, batch_labels in loader:
            batch_inputs = {k: v.to(device, non_blocking=True) for k, v in batch_inputs.items()}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                logits.append(backbone(**batch_inputs).float().cpu())
            labels.append(batch_labels.float())
    probs = torch.sigmoid(torch.cat(logits)).numpy()
    labels = torch.cat(labels).numpy()
    return f1_score(labels, (probs > 0.5).astype(int), average='macro', zero_division=0)

for epoch in range(start_epoch, epochs):
    backbone.train()
    total_loss = 0.0
    for batch_inputs, batch_labels in train_loader:
        batch_inputs = {k: v.to(device, non_blocking=True) for k, v in batch_inputs.items()}
        batch_labels = batch_labels.float().to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            loss = criterion(backbone(**batch_inputs), batch_labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(backbone.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    score = evaluate(test_loader)
    torch.save({
        'epoch': epoch,
        'model_state_dict': backbone.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'target_names': target_names,
        'classes': ALL_5_CLASSES,
    }, checkpoint_path)
    print(f'epoch {epoch + 1:02d}  loss {total_loss / len(train_loader):.4f}  f1_macro {score:.4f}  checkpoint saved')

epoch 01  loss 0.5941  f1_macro 0.3661  checkpoint saved
epoch 02  loss 0.5528  f1_macro 0.4690  checkpoint saved
epoch 03  loss 0.5323  f1_macro 0.5073  checkpoint saved
epoch 04  loss 0.5190  f1_macro 0.5604  checkpoint saved
epoch 05  loss 0.5086  f1_macro 0.5814  checkpoint saved


In [15]:
backbone.eval()
all_logits = []
with torch.no_grad():
    for batch_inputs, _ in test_loader:
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        all_logits.append(backbone(**batch_inputs).cpu())
preds_bin = (torch.sigmoid(torch.cat(all_logits)).numpy() > 0.5).astype(int)
Y_test = Y[test_idx]
print(f"{'Class':<12} {'F1':>7}")
print('-' * 21)
for i, cls in enumerate(ALL_5_CLASSES):
    print(f'{cls:<12} {f1_score(Y_test[:, i], preds_bin[:, i], zero_division=0):>7.4f}')
print('-' * 21)
print(f"{'macro avg':<12} {f1_score(Y_test, preds_bin, average='macro', zero_division=0):>7.4f}")

Class             F1
---------------------
fruity        0.8240
sweet         0.6525
sulfurous     0.5248
woody         0.3221
fatty         0.5830
---------------------
macro avg     0.5813


In [16]:
torch.save({
    'model_state_dict': backbone.state_dict(),
    'target_names': target_names,
    'classes': ALL_5_CLASSES,
}, 'odor_lora.pt')
print('Saved odor_lora.pt')
from google.colab import files
files.download('odor_lora.pt')

Saved odor_lora.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>